# Exercise 1: Image Preprocessing with openEO

This exercise introduces openEO, a platform that lets you request and process satellite imagery through Python code instead of downloading raw files and processing them locally. You will use openEO to compare two processing levels of Sentinel 2 data, apply simple band math, and build a cloud free composite image known as a Best Available Pixel composite.

## What you will learn

1. How to connect to an openEO backend and describe an area and period of interest
2. The practical difference between Sentinel 2 Level 1C and Level 2A products
3. How to perform band math directly on openEO data cubes, using NDVI as an example
4. How to build a Best Available Pixel composite by masking clouds and reducing a time series to a single clean image

## Study area

The exercises below use a small area around Enschede in the Netherlands. The Dutch climate produces frequent cloud cover, which makes this a good location to demonstrate why a single satellite scene is often not usable and why compositing multiple scenes over time is necessary.

## Setting up the connection

openEO is installed as a regular Python package. The first step in any openEO workflow is to connect to a processing backend and authenticate. We use the Copernicus Data Space Ecosystem (CDSE) backend, which provides free access to the full Sentinel 2 archive.

In [ ]:
!pip install openeo rasterio matplotlib numpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

import openeo
from openeo.extra.spectral_indices import compute_indices

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

## Defining the area and period of interest

An openEO request always needs a spatial extent and a temporal extent. The spatial extent is a bounding box given as west, south, east and north coordinates in degrees. The temporal extent is a start and end date.

We define a small bounding box around Enschede and two temporal extents: a single month used for the Level 1C versus Level 2A comparison, and a longer three month window used later for the composite.

In [ ]:
extent = {"west": 6.85, "south": 52.20, "east": 6.92, "north": 52.25}

single_scene_period = ["2023-07-01", "2023-07-31"]
composite_period = ["2023-06-01", "2023-08-31"]

rgb_bands = ["B04", "B03", "B02"]

## Level 1C versus Level 2A

Sentinel 2 data is distributed at two processing levels.

Level 1C contains Top Of Atmosphere reflectance. The signal recorded by the satellite still contains the effect of the atmosphere, so haze, water vapour and aerosols change the apparent brightness and colour of the surface.

Level 2A contains Bottom Of Atmosphere reflectance. An atmospheric correction algorithm called Sen2Cor has already removed the atmospheric contribution, so the values represent surface reflectance more closely. Level 2A products also include a Scene Classification Layer (SCL), a band that labels each pixel as, for example, vegetation, bare soil, water, cloud or cloud shadow. This layer is essential for cloud masking, which we use later in this exercise.

The cell below loads the same area and month from both collections and keeps only the true colour bands, so the two products can be compared directly.

In [ ]:
s2_l1c = connection.load_collection(
    "SENTINEL2_L1C",
    spatial_extent=extent,
    temporal_extent=single_scene_period,
    bands=rgb_bands,
    max_cloud_cover=30,
)

s2_l2a = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=extent,
    temporal_extent=single_scene_period,
    bands=rgb_bands,
    max_cloud_cover=30,
)

# keep the first cloud free observation of the month for each product
l1c_scene = s2_l1c.reduce_dimension(dimension="t", reducer="first")
l2a_scene = s2_l2a.reduce_dimension(dimension="t", reducer="first")

l1c_job = l1c_scene.save_result(format="GTiff")
l2a_job = l2a_scene.save_result(format="GTiff")

openeo.MultiResult([l1c_job, l2a_job]).create_job(title="L1C vs L2A comparison").start_and_wait()

In [ ]:
job = [j for j in connection.list_jobs() if j["title"] == "L1C vs L2A comparison"][0]
batch_job = connection.job(job["id"])
batch_job.get_results().download_files("data/l1c_vs_l2a")

## Comparing the two products visually

Both files were saved with generic file names, since two results were requested from the same job. We list the downloaded files and open the two GeoTIFFs with rasterio to inspect them.

When you look at the two images side by side, pay attention to the overall haze and colour cast in the Level 1C image compared to the Level 2A image. The Level 2A image should appear sharper and more consistent in colour, since the atmospheric contribution has been removed.

In [ ]:
import glob

downloaded_files = sorted(glob.glob("data/l1c_vs_l2a/*.tif"))
print(downloaded_files)

l1c_path, l2a_path = downloaded_files[0], downloaded_files[1]

with rasterio.open(l1c_path) as src:
    l1c_image = src.read([1, 2, 3])

with rasterio.open(l2a_path) as src:
    l2a_image = src.read([1, 2, 3])


def stretch_to_rgb(image, upper_percentile=98):
    """Scale a three band reflectance array to the 0 to 1 range for display."""
    image = np.moveaxis(image, 0, -1).astype(float)
    upper_value = np.percentile(image, upper_percentile)
    image = np.clip(image / upper_value, 0, 1)
    return image


fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(stretch_to_rgb(l1c_image))
axes[0].set_title("Sentinel 2 Level 1C (Top Of Atmosphere)")
axes[1].imshow(stretch_to_rgb(l2a_image))
axes[1].set_title("Sentinel 2 Level 2A (Bottom Of Atmosphere)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Simple band math

A data cube in openEO behaves like a numeric array. Individual bands can be selected and combined with ordinary arithmetic operators, which makes it straightforward to compute a vegetation index by hand.

The Normalized Difference Vegetation Index (NDVI) compares reflectance in the near infrared band, where healthy vegetation reflects strongly, with reflectance in the red band, where vegetation absorbs light for photosynthesis. The index is defined as:

NDVI = (NIR minus Red) divided by (NIR plus Red)

The cell below loads the red and near infrared bands and computes NDVI directly with the standard Python arithmetic operators. No dedicated index function is required for a calculation this simple.

In [ ]:
s2_for_index = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=extent,
    temporal_extent=single_scene_period,
    bands=["B04", "B08"],
    max_cloud_cover=30,
)

s2_for_index = s2_for_index.reduce_dimension(dimension="t", reducer="first")

red_band = s2_for_index.band("B04")
nir_band = s2_for_index.band("B08")

ndvi_manual = (nir_band - red_band) / (nir_band + red_band)

ndvi_manual.save_result(format="GTiff").create_job(title="NDVI manual band math").start_and_wait()
job = [j for j in connection.list_jobs() if j["title"] == "NDVI manual band math"][0]
connection.job(job["id"]).get_results().download_files("data/ndvi_manual")

openEO also provides a helper function called `compute_indices`, which computes a large number of standard vegetation and water indices from a data cube without requiring the user to write the formula. It is useful once you are comfortable with band math and want to compute several indices quickly. The cell below computes the same NDVI value with this helper, so the two approaches can be compared.

In [ ]:
s2_for_index_helper = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=extent,
    temporal_extent=single_scene_period,
    bands=["B04", "B08"],
    max_cloud_cover=30,
)
s2_for_index_helper = s2_for_index_helper.reduce_dimension(dimension="t", reducer="first")

ndvi_helper = compute_indices(s2_for_index_helper, indices=["NDVI"])
ndvi_helper.save_result(format="GTiff").create_job(title="NDVI compute indices").start_and_wait()
job = [j for j in connection.list_jobs() if j["title"] == "NDVI compute indices"][0]
connection.job(job["id"]).get_results().download_files("data/ndvi_helper")

In [ ]:
ndvi_manual_file = glob.glob("data/ndvi_manual/*.tif")[0]
ndvi_helper_file = glob.glob("data/ndvi_helper/*.tif")[0]

with rasterio.open(ndvi_manual_file) as src:
    ndvi_manual_array = src.read(1)

with rasterio.open(ndvi_helper_file) as src:
    ndvi_helper_array = src.read(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(ndvi_manual_array, cmap="RdYlGn", vmin=-1, vmax=1)
axes[0].set_title("NDVI from manual band math")
im1 = axes[1].imshow(ndvi_helper_array, cmap="RdYlGn", vmin=-1, vmax=1)
axes[1].set_title("NDVI from compute_indices")
for ax in axes:
    ax.axis("off")
fig.colorbar(im0, ax=axes, fraction=0.03, pad=0.02, label="NDVI")
plt.show()

print("Maximum absolute difference between the two methods:",
      np.nanmax(np.abs(ndvi_manual_array - ndvi_helper_array)))

## Building a Best Available Pixel composite

A single Sentinel 2 scene is frequently affected by clouds, especially in a region such as the Netherlands. Rather than accepting whatever a single date happens to show, it is common practice in remote sensing to combine several scenes acquired over a period of weeks or months into one composite image that shows only clear, cloud free land surface. This is generally referred to as a Best Available Pixel composite, or BAP composite.

The general recipe is as follows. For every date in the period of interest, look at the corresponding Scene Classification Layer and mark the pixels that are cloud, cloud shadow or cirrus. Remove those pixels from the data. Then, for every pixel location, summarise whatever clear observations remain across the whole period into a single value, for example by taking the median.

The function below implements the masking step. It labels SCL classes 3 (cloud shadow), 8 (cloud, medium probability), 9 (cloud, high probability) and 10 (thin cirrus) as unusable and removes them from the corresponding reflectance data.

In [ ]:
def mask_clouds(scl_cube, data_cube):
    """Remove cloud, cloud shadow and cirrus pixels from a data cube using the SCL band."""
    cloud_classes = (scl_cube == 3) | (scl_cube == 8) | (scl_cube == 9) | (scl_cube == 10)
    return data_cube.mask(cloud_classes)

We now load three months of Sentinel 2 Level 2A data for the true colour bands, together with the matching SCL band, mask out clouds, and reduce the resulting time series to a single image using the median. The median is a simple and robust choice for this step, since it is not affected much by a small number of remaining bad pixels, unlike the mean.

In [ ]:
s2_series = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=extent,
    temporal_extent=composite_period,
    bands=rgb_bands,
    max_cloud_cover=80,
)

scl_series = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=extent,
    temporal_extent=composite_period,
    bands=["SCL"],
    max_cloud_cover=80,
)

s2_series_masked = mask_clouds(scl_series.band("SCL"), s2_series)
bap_composite = s2_series_masked.reduce_dimension(dimension="t", reducer="median")

bap_composite.save_result(format="GTiff").create_job(title="BAP composite").start_and_wait()
job = [j for j in connection.list_jobs() if j["title"] == "BAP composite"][0]
connection.job(job["id"]).get_results().download_files("data/bap_composite")

## Comparing a single scene with the composite

The final step is to compare the single Level 2A scene used earlier in this exercise with the three month Best Available Pixel composite. Cloud and haze artefacts visible in the single scene should be largely absent from the composite, and the land surface should appear more uniform.

In [ ]:
bap_file = glob.glob("data/bap_composite/*.tif")[0]

with rasterio.open(bap_file) as src:
    bap_image = src.read([1, 2, 3])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(stretch_to_rgb(l2a_image))
axes[0].set_title("Single Level 2A scene, July 2023")
axes[1].imshow(stretch_to_rgb(bap_image))
axes[1].set_title("Best Available Pixel composite, June to August 2023")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Summary

This exercise covered three practical building blocks that recur throughout satellite based mapping workflows.

Level 1C and Level 2A products differ in whether the atmosphere has been corrected for. Level 2A is generally the better starting point for quantitative analysis, since surface reflectance is what most vegetation and water indices assume as input.

Band math can be written directly with arithmetic operators on openEO data cubes. This gives full control over the formula being used and is a good starting point before turning to helper functions such as `compute_indices` for convenience.

A Best Available Pixel composite combines a masking step, which removes unreliable pixels using the Scene Classification Layer, with a reducing step, which summarises the remaining observations into a single representative image. This same masking and reducing pattern reappears in many other Earth observation workflows, including cloud free change detection and time series analysis.